# HomeLLM LLaMA-style: Module Playground

Низкоуровневый notebook для экспериментов:
- напрямую работаем с `nn.Module`
- меняем архитектуру через `HomeConfig`
- запускаем ручной train-loop с `tqdm.notebook`
- можно подменять/добавлять свои блоки и лоссы

In [ ]:
from pathlib import Path
import json
import math

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

from homellm.models.home_model import HomeConfig, HomeForCausalLM
from homellm.training.pretrain import StreamingTextDataset

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

In [ ]:
# ===== Paths and runtime =====
PROJECT_ROOT = Path('/app')
DATA_PATH = PROJECT_ROOT / 'datasets' / 'fineweb-2_train.jsonl'  # поменяй при необходимости
ensure_pretrain_dataset(str(DATA_PATH))  # скачает с HF, если файла нет
OUT_DIR = PROJECT_ROOT / 'out' / 'playground'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print('data:', DATA_PATH)
print('out:', OUT_DIR)
print('device:', DEVICE, 'dtype:', DTYPE)

In [ ]:
# ===== Tokenizer + dataset =====
TOKENIZER_ID = 'gpt2'
SEQ_LEN = 2048
BATCH_SIZE = 2

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|pad|>'})

dataset = StreamingTextDataset(str(DATA_PATH), tokenizer, seq_len=SEQ_LEN)
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=collator, num_workers=0)

print('tokenizer vocab:', len(tokenizer))

In [ ]:
# ===== Build base HomeModel (LLaMA-style) as nn.Module =====
# Тут меняй любые параметры архитектуры
cfg = HomeConfig(
    vocab_size=len(tokenizer),
    hidden_size=512,
    num_hidden_layers=8,
    num_attention_heads=8,
    max_position_embeddings=SEQ_LEN,
    use_sdpa=True,
    use_liger=True,
)
base_model = HomeForCausalLM(cfg)
base_model.resize_token_embeddings(len(tokenizer))

total_params = sum(p.numel() for p in base_model.parameters())
print(f'params: {total_params:,}')

In [ ]:
# ===== Example: your own custom nn.Module wrapper =====
# Пользователь может здесь менять forward как угодно.
class MyHomeModule(nn.Module):
    def __init__(self, model: HomeForCausalLM):
        super().__init__()
        self.model = model
        # Пример кастомного trainable gate (можно удалить/заменить)
        self.logit_scale = nn.Parameter(torch.tensor(1.0))

    def forward(self, input_ids, attention_mask=None, labels=None):
        out = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        logits = out.logits * self.logit_scale
        loss = out.loss
        if labels is not None:
            # Кастомный CE (пример того, как переопределять лосс)
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = nn.functional.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )
        return {'loss': loss, 'logits': logits}

model = MyHomeModule(base_model).to(DEVICE)
print('custom module ready')

In [ ]:
# ===== Optimizer / training config =====
LR = 3e-4
WEIGHT_DECAY = 0.1
MAX_STEPS = 200  # для быстрых экспериментов
GRAD_ACCUM = 4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler_enabled = torch.cuda.is_available()  # для bf16/fp16 autocast
print('max_steps:', MAX_STEPS)

In [ ]:
# ===== Manual training loop with tqdm =====
model.train()
optimizer.zero_grad(set_to_none=True)

loss_history = []
pbar = tqdm(total=MAX_STEPS, desc='playground-train')

step = 0
for batch in loader:
    input_ids = batch['input_ids'].to(DEVICE)
    attention_mask = batch.get('attention_mask')
    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)
    labels = batch['labels'].to(DEVICE)

    with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = out['loss'] / GRAD_ACCUM

    loss.backward()

    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    step_loss = float(loss.detach().item() * GRAD_ACCUM)
    loss_history.append(step_loss)
    pbar.set_postfix({'loss': f'{step_loss:.4f}'})
    pbar.update(1)
    step += 1

    if step >= MAX_STEPS:
        break

pbar.close()
print('done, final loss:', loss_history[-1] if loss_history else None)

In [ ]:
# ===== Save artifacts =====
final_dir = OUT_DIR / 'module_playground_final'
final_dir.mkdir(parents=True, exist_ok=True)

model.model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

with open(final_dir / 'loss_history.json', 'w', encoding='utf-8') as f:
    json.dump(loss_history, f, ensure_ascii=False, indent=2)

print('saved to', final_dir)